In [2]:
pip install pymupdf

Note: you may need to restart the kernel to use updated packages.


In [6]:
import os
import fitz  # PyMuPDF

# =====================================================================
# --- PREPRESS GEOMETRY CONFIGURATION (1 inch = 72 points) ---
# =====================================================================
PT = 72.0

# Page Dimensions (10.625" x 17.125")
PAGE_W = 10.625 * PT   # 765.0 pt
PAGE_H = 17.125 * PT   # 1233.0 pt

# Sheet Dimensions (29" x 23" Landscape)
SHEET_W = 29.0 * PT    # 2088.0 pt
SHEET_H = 23.0 * PT    # 1656.0 pt

# Calculate exact centering offsets for symmetrical press margins
TOTAL_CLUSTER_W = PAGE_H + PAGE_W         # 1233 + 765 = 1998 pt (27.75")
OFFSET_X = (SHEET_W - TOTAL_CLUSTER_W) / 2 # 45.0 pt (0.625" side margins)

# Vertically center each column independently on the 23" sheet
COL1_H = PAGE_W * 2                       # 1530 pt (two 10.625" sides stacked)
COL1_Y = (SHEET_H - COL1_H) / 2           # 63.0 pt top/bottom margin for Pos 1 & 2

COL2_H = PAGE_H                           # 1233 pt
COL2_Y = (SHEET_H - COL2_H) / 2           # 211.5 pt top/bottom margin for Pos 3


def impose_cut_and_stack(input_pdf_path, output_pdf_path):
    print(f"Opening source file: {input_pdf_path}")
    source_doc = fitz.open(input_pdf_path)
    total_pages = len(source_doc)
    
    # 1. PAIR PAGES INTO RECORDS (Page 0=Front, Page 1=Back)
    records = []
    for i in range(0, total_pages, 2):
        front_idx = i
        back_idx = i + 1 if (i + 1) < total_pages else None
        records.append({'front': front_idx, 'back': back_idx})
        
    total_records = len(records)
    total_sheets = (total_records + 2) // 3
    print(f"Loaded {total_records} records. Generating {total_sheets} press sheets (Front/Back)...")
    
    # Pad records list if not evenly divisible by 3
    while len(records) % 3 != 0:
        records.append({'front': None, 'back': None})
        
    # 2. SPLIT INTO CUT-AND-STACK STREAMS
    # Slicing ensures Stack 1 gets Recs 1..N, Stack 2 gets N+1..2N, etc.
    stack1 = records[0:total_sheets]
    stack2 = records[total_sheets : 2*total_sheets]
    stack3 = records[2*total_sheets : 3*total_sheets]
    
    # 3. BUILD IMPOSED MASTER PDF
    output_doc = fitz.open()
    
    for s in range(total_sheets):
        # -------------------------------------------------------------
        # FRONT SIDE OF PRESS SHEET
        # -------------------------------------------------------------
        sheet_front = output_doc.new_page(width=SHEET_W, height=SHEET_H)
        
        # Position 1 Front (Top-Left Column, Rotated 90°)
        if stack1[s]['front'] is not None:
            rect = fitz.Rect(OFFSET_X, COL1_Y, OFFSET_X + PAGE_H, COL1_Y + PAGE_W)
            sheet_front.show_pdf_page(rect, source_doc, stack1[s]['front'], rotate=90)
            
        # Position 2 Front (Bottom-Left Column, Rotated 90°)
        if stack2[s]['front'] is not None:
            rect = fitz.Rect(OFFSET_X, COL1_Y + PAGE_W, OFFSET_X + PAGE_H, COL1_Y + (2 * PAGE_W))
            sheet_front.show_pdf_page(rect, source_doc, stack2[s]['front'], rotate=90)
            
        # Position 3 Front (Right Column, Portrait 0°)
        if stack3[s]['front'] is not None:
            rect = fitz.Rect(OFFSET_X + PAGE_H, COL2_Y, OFFSET_X + TOTAL_CLUSTER_W, COL2_Y + PAGE_H)
            sheet_front.show_pdf_page(rect, source_doc, stack3[s]['front'], rotate=0)

        # -------------------------------------------------------------
        # BACK SIDE OF PRESS SHEET (Mirrored Horizontally for Duplexing)
        # -------------------------------------------------------------
        sheet_back = output_doc.new_page(width=SHEET_W, height=SHEET_H)
        
        # Position 3 Back (Mirrored to LEFT Column, Portrait 0°)
        if stack3[s]['back'] is not None:
            rect = fitz.Rect(OFFSET_X, COL2_Y, OFFSET_X + PAGE_W, COL2_Y + PAGE_H)
            sheet_back.show_pdf_page(rect, source_doc, stack3[s]['back'], rotate=0)
            
        # Position 1 Back (Mirrored to Top-RIGHT Column, Rotated 270° for head alignment)
        if stack1[s]['back'] is not None:
            rect = fitz.Rect(OFFSET_X + PAGE_W, COL1_Y, OFFSET_X + TOTAL_CLUSTER_W, COL1_Y + PAGE_W)
            sheet_back.show_pdf_page(rect, source_doc, stack1[s]['back'], rotate=270)
            
        # Position 2 Back (Mirrored to Bottom-RIGHT Column, Rotated 270°)
        if stack2[s]['back'] is not None:
            rect = fitz.Rect(OFFSET_X + PAGE_W, COL1_Y + PAGE_W, OFFSET_X + TOTAL_CLUSTER_W, COL1_Y + (2 * PAGE_W))
            sheet_back.show_pdf_page(rect, source_doc, stack2[s]['back'], rotate=270)

    # 4. SAVE FINAL FILE
    os.makedirs(os.path.dirname(output_pdf_path), exist_ok=True)
    output_doc.save(output_pdf_path, garbage=3, deflate=True)
    output_doc.close()
    source_doc.close()
    print(f"Success! Imposed press file saved to:\n -> {output_pdf_path}")


# =====================================================================
# --- EXECUTION BLOCK ---
# =====================================================================
if __name__ == "__main__":
    
    # Put 'r' in front of paths so Windows backslashes work cleanly
    INPUT_PDF = r"C:\Users\8DATA2\Desktop\Working Folder\Bastard Cut\7-1-2026 Rickey Pete Voter Guide 1UP BATCH_1.pdf"
    
    # Path where you want the single output PDF saved
    OUTPUT_PDF = r"C:\Users\8DATA2\Desktop\Working Folder\Bastard Cut\7-1-2026 Rickey Pete Voter Guide 1UP BATCH_1 IMPOSED.pdf"
    
    impose_cut_and_stack(INPUT_PDF, OUTPUT_PDF)

Opening source file: C:\Users\8DATA2\Desktop\Working Folder\Bastard Cut\7-1-2026 Rickey Pete Voter Guide 1UP BATCH_1.pdf
Loaded 3000 records. Generating 1000 press sheets (Front/Back)...
Success! Imposed press file saved to:
 -> C:\Users\8DATA2\Desktop\Working Folder\Bastard Cut\7-1-2026 Rickey Pete Voter Guide 1UP BATCH_1 IMPOSED.pdf


In [21]:
import os
import fitz  # PyMuPDF

# =====================================================================
# --- PREPRESS GEOMETRY CONFIGURATION (1 inch = 72 points) ---
# =====================================================================
PT = 72.0

# Page Dimensions (10.625" x 17.125")
PAGE_W = 10.625 * PT   # 765.0 pt
PAGE_H = 17.125 * PT   # 1233.0 pt

# Sheet Dimensions (23" x 29" Portrait)
SHEET_W = 23.0 * PT    # 1656.0 pt
SHEET_H = 29.0 * PT    # 2088.0 pt

# Vertical Centering (Top block + Bottom block)
TOTAL_CLUSTER_H = PAGE_H + PAGE_W          # 1233 + 765 = 1998 pt (27.75")
OFFSET_Y = (SHEET_H - TOTAL_CLUSTER_H) / 2 # 45.0 pt (0.625" top/bottom margin)

# Horizontal Centering for Top Block (Positions 1 & 2 Side-by-Side)
TOP_BLOCK_W = PAGE_W * 2                   # 1530 pt (21.25")
OFFSET_X_TOP = (SHEET_W - TOP_BLOCK_W) / 2 # 63.0 pt (0.875" side margins)

# Horizontal Centering for Bottom Block (Position 3 Rotated Sideways)
BOTTOM_BLOCK_W = PAGE_H                    # 1233 pt (17.125")
OFFSET_X_BOTTOM = (SHEET_W - BOTTOM_BLOCK_W) / 2 # 211.5 pt (2.9375" side margins)


def impose_cut_and_stack_portrait(input_pdf_path, output_pdf_path):
    print(f"Opening source file: {input_pdf_path}")
    source_doc = fitz.open(input_pdf_path)
    total_pages = len(source_doc)
    
    # 1. PAIR PAGES INTO RECORDS (Page 0=Front, Page 1=Back)
    records = []
    for i in range(0, total_pages, 2):
        front_idx = i
        back_idx = i + 1 if (i + 1) < total_pages else None
        records.append({'front': front_idx, 'back': back_idx})
        
    total_records = len(records)
    total_sheets = (total_records + 2) // 3
    print(f"Loaded {total_records} records. Generating {total_sheets} portrait press sheets...")
    
    while len(records) % 3 != 0:
        records.append({'front': None, 'back': None})
        
    # 2. SPLIT INTO CUT-AND-STACK STREAMS
    stack1 = records[0:total_sheets]
    stack2 = records[total_sheets : 2*total_sheets]
    stack3 = records[2*total_sheets : 3*total_sheets]
    
    # 3. BUILD IMPOSED MASTER PDF
    output_doc = fitz.open()
    
    for s in range(total_sheets):
        # -------------------------------------------------------------
        # FRONT SIDE OF PRESS SHEET
        # -------------------------------------------------------------
        sheet_front = output_doc.new_page(width=SHEET_W, height=SHEET_H)
        
        # Position 1 Front (Top-Left, Upright 0°)
        if stack1[s]['front'] is not None:
            rect = fitz.Rect(OFFSET_X_TOP, OFFSET_Y, OFFSET_X_TOP + PAGE_W, OFFSET_Y + PAGE_H)
            sheet_front.show_pdf_page(rect, source_doc, stack1[s]['front'], rotate=0)
            
        # Position 2 Front (Top-Right, Upright 0°)
        if stack2[s]['front'] is not None:
            rect = fitz.Rect(OFFSET_X_TOP + PAGE_W, OFFSET_Y, OFFSET_X_TOP + TOP_BLOCK_W, OFFSET_Y + PAGE_H)
            sheet_front.show_pdf_page(rect, source_doc, stack2[s]['front'], rotate=0)
            
        # Position 3 Front (Bottom-Center, Rotated 90° Clockwise)
        if stack3[s]['front'] is not None:
            rect = fitz.Rect(OFFSET_X_BOTTOM, OFFSET_Y + PAGE_H, OFFSET_X_BOTTOM + BOTTOM_BLOCK_W, OFFSET_Y + TOTAL_CLUSTER_H)
            sheet_front.show_pdf_page(rect, source_doc, stack3[s]['front'], rotate=90)

        # -------------------------------------------------------------
        # BACK SIDE OF PRESS SHEET (Mirrored Horizontally for Duplexing)
        # -------------------------------------------------------------
        sheet_back = output_doc.new_page(width=SHEET_W, height=SHEET_H)
        
        # Position 1 Back (Mirrored to Top-RIGHT, Upright 0°)
        if stack1[s]['back'] is not None:
            rect = fitz.Rect(OFFSET_X_TOP + PAGE_W, OFFSET_Y, OFFSET_X_TOP + TOP_BLOCK_W, OFFSET_Y + PAGE_H)
            sheet_back.show_pdf_page(rect, source_doc, stack1[s]['back'], rotate=0)
            
        # Position 2 Back (Mirrored to Top-LEFT, Upright 0°)
        if stack2[s]['back'] is not None:
            rect = fitz.Rect(OFFSET_X_TOP, OFFSET_Y, OFFSET_X_TOP + PAGE_W, OFFSET_Y + PAGE_H)
            sheet_back.show_pdf_page(rect, source_doc, stack2[s]['back'], rotate=0)
            
        # Position 3 Back (Bottom-Center, Rotated 270° for Head Alignment)
        # Horizontal placement stays the same because it's perfectly centered
        if stack3[s]['back'] is not None:
            rect = fitz.Rect(OFFSET_X_BOTTOM, OFFSET_Y + PAGE_H, OFFSET_X_BOTTOM + BOTTOM_BLOCK_W, OFFSET_Y + TOTAL_CLUSTER_H)
            sheet_back.show_pdf_page(rect, source_doc, stack3[s]['back'], rotate=270)

    # 4. SAVE FINAL FILE
    os.makedirs(os.path.dirname(output_pdf_path), exist_ok=True)
    output_doc.save(output_pdf_path, garbage=3, deflate=True)
    output_doc.close()
    source_doc.close()
    print(f"Success! Portrait imposed file saved to:\n -> {output_pdf_path}")


# =====================================================================
# --- EXECUTION BLOCK ---
# =====================================================================
if __name__ == "__main__":
    
    INPUT_PDF = r"C:\Users\8DATA2\Desktop\Working Folder\Bastard Cut\122756 Rickey Pete Voter Guide 1UP BATCH_10.pdf"
    OUTPUT_PDF = r"C:\Users\8DATA2\Desktop\Working Folder\Bastard Cut\122756 Rickey Pete Voter Guide 1UP BATCH_10 IMPOSED.pdf"
    
    impose_cut_and_stack_portrait(INPUT_PDF, OUTPUT_PDF)

Opening source file: C:\Users\8DATA2\Desktop\Working Folder\Bastard Cut\122756 Rickey Pete Voter Guide 1UP BATCH_10.pdf
Loaded 3000 records. Generating 1000 portrait press sheets...
Success! Portrait imposed file saved to:
 -> C:\Users\8DATA2\Desktop\Working Folder\Bastard Cut\122756 Rickey Pete Voter Guide 1UP BATCH_10 IMPOSED.pdf
